## Diploma Thesis: Multi-Block Replacement

The objective of this notebook is to extend work on introduction notebook. Here we will focus on create an MVP methodology on how to replace multiple blocks inside an input transformer model.  
The key difference is that in the case of multi-block replacements and error propagation occurs across the model (Grafting paper). To ensure efficiency we utilize again the calibration data  
for local scope working as well as knowledge distillation for fine-tune retraining.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
import pprint
from typing import Literal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
project_pwd = Path().cwd().parents[1]
project_pwd = os.path.abspath(project_pwd)
if project_pwd not in sys.path:
    sys.path.append(project_pwd)

sys.dont_write_bytecode = True

In [3]:
from scripts.intro.model import *
from scripts.intro.loader import *
from scripts.intro.block import *
from scripts.intro.activation import *
from scripts.intro.replacement import *
from scripts.intro.eval import *
from scripts.intro.metrics import *

from scripts.utils import *
from configs.intro_config import *

Model config - extended to multi-block workflow

In [4]:
mcfg = MCFG()
torch.manual_seed(mcfg.seed)

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

### Workflow

1) Load model and tokenizer
2) make data loader
3) collect blocks i/o pairs (per block)
4) evaluate block importance (per block)
5) select replacement strategy
6) apply replacement strategy on blocks
7) fine-tuning based on replacement strategy
8) eval

___

model utils

In [6]:
def layer_idx_to_mlp_path(model, layer_idx):
    # NOTE: namapuje index MLP bloku na meno bloku v modely
    paths = list_mlp_paths(model)
    expected = f"model.layers.{layer_idx}.mlp"

    if expected in paths:
        return expected
    
    matches = []
    for p in paths:
        if p.endswith(".mlp") and f".{layer_idx}.mlp" in p:
            matches.append(p)

    if not matches:
        raise ValueError("bad block index provided.")
    return matches[0]

block scoring

In [7]:
def bi_cosine_score(X, Y):
    # NOTE: Minitron paper metrika (BI skore)
    cos = F.cosine_similarity(X, Y, dim=-1, eps=1e-8)
    score = 1.0 - cos
    score = score.mean().item()
    return float(score)


def compute_bi_scores(model, layer_idxs, loader, cfg, device):
    bi_scores = {}
    for idx in layer_idxs:
        idx = int(idx)
        path = layer_idx_to_mlp_path(model, idx)
        X, Y = collect_block_io(model, path, loader, cfg.num_calib_batches, device)
        bi_scores[idx] = bi_cosine_score(X, Y)

    return bi_scores

vyber operatora

In [22]:
def fit_replacement_operator(operator_name, X, Y, hidden_size, device, cfg):
    if operator_name == "linear":
        return fit_linear_replacement(X, Y, hidden_size, device, cfg)
    
    raise ValueError(f"Unsupported operator, operator={operator_name}")

recovery funkcie

In [16]:
def global_recovery_finetunning(model, calib_loader, cfg, device):
    # TODO
    return model


def local_recovery_finetunning(model, calib_loader, cfg, device):
    # TODO
    return model

replacement strategy

In [4]:
def run_one_shot_replacement(model, target_layer_idxs, calib_loader, eval_loader, cfg, device):
    target_paths = {}
    for i in target_layer_idxs:
        target_paths[i] = layer_idx_to_mlp_path(model, layer_idx=i)
    
    replacements = {}
    logs = []

    # NOTE: zatial linear
    operator = 'linear'

    for i, path in target_paths.items():
        X_train, Y_train = collect_block_io(model, path, calib_loader, cfg.num_calib_batches, device)
        X_val, Y_val = collect_block_io(model, path, eval_loader, cfg.num_eval_batches, device)
        
        hidden_size = X_train.shape[-1]
        replacement_operator = fit_replacement_operator(operator, X_train, Y_train, hidden_size, device, cfg)
        
        # NOTE: validacia cez MSE mozno skusim aj nejake ine loss funkcie
        new_operator_mse = evaluate_block_mse(replacement_operator, X_val, Y_val, device)
        
        log = {
            "layer_idx" : i,
            "layer_path" : path,
            "new_operator_mse" : new_operator_mse
        }
        
        logs.append(log)
        replacements[i] = replacement_operator

    for i in target_layer_idxs:
        replace_submodule(model, target_paths[i], replacements[i].to(device))

    model = global_recovery_finetunning(model, calib_loader, cfg, device)
    model_loss, model_ppl = evaluate_lm(model, eval_loader, cfg.num_eval_batches, device)

    data = {
        "strategy" : cfg.replacement_strategy,
        "model_loss" : model_loss,
        "model_ppl" : model_ppl,
        "logs" : logs
    }

    return data

In [13]:
def run_iterative_replacement(model, target_layer_idxs, calib_loader, eval_loader, cfg, device):
    target_paths = {}
    for i in target_layer_idxs:
        target_paths[i] = layer_idx_to_mlp_path(model, layer_idx=i)
    
    replacements = {}
    logs = []

    # NOTE: zatial linear
    operator = 'linear'

    for i, path in target_paths.items():
        X_train, Y_train = collect_block_io(model, path, calib_loader, cfg.num_calib_batches, device)
        X_val, Y_val = collect_block_io(model, path, eval_loader, cfg.num_eval_batches, device)

        hidden_size = X_train.shape[-1]
        replacement = fit_replacement_operator(operator, X_train, Y_train, hidden_size, device, cfg)
        
        # NOTE: validacia cez MSE mozno skusim aj nejake ine loss funkcie
        new_operator_mse = evaluate_block_mse(replacement, X_val, Y_val, device)

        replace_submodule(model, path, replacement.to(device))
        model = local_recovery_finetunning(model, calib_loader, cfg, device)

        iter_loss, iter_ppl = evaluate_lm(model, eval_loader, cfg.num_eval_batches, device)

        log = {
            "layer_idx" : i,
            "path" : path,
            "new_operator_mse" : new_operator_mse,
            "iter_loss"  : iter_loss,
            "iter_ppl" : iter_ppl
        }

        logs.append(log)

    data = {
        "strategy" : cfg.replacement_strategy,
        "final_loss" : logs[-1]["iter_loss"],
        "final_ppl" : logs[-1]["iter_ppl"],
        "logs" : logs
    }
    
    return data

search

In [10]:
def one_shot_search_replacement():
    # NOTE: toto sa odlisuje od iterativu tak ze robime iteracie nad k z original modelu
    # iterative po kazdej iteracii berie pruned model, zatial co tu berieme kolko blokov chceme
    # nahradit z original modelu (napr. 1, 2, ..., k)

    # cize robime one_shot replacement v style for k in range(1, max_k) a zapisujeme vysledky do dict
    pass

grafting sequential replacement (kazdy druhy blok)
- moze sluzit ako metodicky baseline pre porovnanie prunning vysledkov

In [12]:
def grafting_sequenntial_replacement():
    # NOTE: podla grafting paperu nahradit kazdy druhy blok
    pass

___
### Data Load

___

### Testovanie experimentov

#### Methodology

We have taken inspiration from the source papers (Minitron Compact Language Models, Grafting) to use the following methodological steps:

1. Architecture:
    1) local KD on block -> operator to operator
    2) model-level fine-tuning -> model error fix
    3) use KD, L2 mse for block KD - Grafting
    4) use KD, KL divergence for model wide - Minitron

    5) Depth prunning -> nevyhadzujeme operatory kompletne ale ak nieco tak nahrada
    


random-k vs top-k replacement

one-shot vs iterative

expansion ratio

___
### Analyza

___

### Self-Notes

Otazky:

- Ako urcit dostatocne dobre k? Zatial to bolo manualne
- Treba robit selekciu operatorov rucne alebo nejakym rozumnym sposobom vieme automatizovat a spravit vztah medzi BI score a operatorom?

Notes:

- to co som robil tu v MVP mi pride dost taky blind-search, okrem BI informacie
to je skor trial and error skusanie, v priebehu vyskumu sa chcem sutredovat aj na interpretaciu spravania sa modelu, vrstiev, blokov, ... na zaklade toho rozhodovat

- buduci design bude asi focusovat urcite layers abstrakcie:
    - automatizacia: loading datasetu, zbieranie kalibracnych dat, replacement bloku, ...
    - screening: tu asi prebehne nejaky styl scoringu blokov/replacement navrhov, vyhodnotenie strategie + logging -> v zasade je asi dobre postavit nejaky dobry monitoring nad celou pipelineou nech mame model specific behaviour
    - chirurgia: na zaklade screeningu ak stanovime nejky config policy mozeme robit nasledovne:
        - definovat na zaklade nejake policy plne automatizovany replacement (na subsete modelu)
        - naprogramovat nejake dalsie funkcie ktore pomozu robit lahky manualny replacement na tych konkretnych blokoch kde chceme


    - otazka je asi ze ako dobre nadefinovat taku strategiu (zatial ma napady asi ze spravit dict: layer -> operator)
    - ak by sme robili dict: layer -> operator tak potom na zaklade screeningovych logov vieme spravit aj block analyzu a spravit custom operator (napr na zaklade aktivacii, gradientov, ...)
        - to mi dojde asi najrozumnejsie v pripade hybridov (e.g. Linear + MLP), lebo bude zalezat ktore vrstvy operatora su tie podstatne ktore vyzaduju nelinearitu

Hybridne metody:

- da sa nejako MLP na zaklade importance bloku, importance MLP vrstviev a importance jednotlivych neuronov achitektonicky poskladat hybridny replacement?
    - napr. linear layer na subsete low-score vrstiev
    - smaller MLP na subsete high-score vrstiev (papers ukazuju ze su to vacsinou tie posledne)

Ideas (Experimenty):

1. Baselines:
    - k-random vs k-least important block replacements with linear layer -> sledovanie efektu degradacie modelu
    mozno by to mohlo byt useful (idea je ist random vs podla dolezitosti operatorov)
    - porovnanie One-Shot vs Iterative replacement: podla Minitron paperu je One-Shot kontraintuitivne lepsi (zaujimave), cielom tu je sa pozriet ci je to pravda aj v pripade cisto len MLP blok nahrad (porovnat aj rozne vypoctove budgety pri One-Shot)

2. Architektury:
    - porovnat pure NAS (Neural search architecture), teda cela architektura pruned modelu cez NAS vs partial NAS, ze napr cez policy BI < 0.1 budu vsetky bloky replaced nejakou linear layer (popr. nieco co bude good simple replacement) a tie najdolezitejsie pojdu cez NAS

    - budovanie hybridu na zaklade aktivacii vrstiev v bloku: teda ak mam blok $l$ tak jeho vrstvy a neurony maju importance -> na zaklade toho postavit hybrid co je a neni important